# Fused Triton kernels: run everything on a Colab GPU

This notebook reproduces every result in the repo
[JawadurRahman/triton-fused-kernels](https://github.com/JawadurRahman/triton-fused-kernels):

1. check that a GPU is attached,
2. clone the repo and install the few missing dependencies,
3. run the correctness tests on the GPU,
4. run the timing benchmark and the (optional) energy benchmark,
5. make the plots, and save `results/` and `plots/` as a zip.

**Before running:** Runtime → Change runtime type → **T4 GPU**. A full run
takes roughly 15–30 minutes, mostly in the timing sweep and `torch.compile`.
The T4 has no bf16 support, so everything here is fp16 and fp32.

## 1. Check for a GPU

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), (
    "No GPU attached. Use Runtime > Change runtime type > T4 GPU, then rerun."
)
print(torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

## 2. Get the code and install dependencies

Colab already ships `torch` with a matching `triton`, so they are **not**
reinstalled (reinstalling torch on Colab can break its CUDA setup). Only the
NVML bindings for the energy benchmark, plus anything else missing, are installed.

In [ ]:
import os
REPO = "/content/triton-fused-kernels"
if not os.path.exists(REPO):
    !git clone https://github.com/JawadurRahman/triton-fused-kernels.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
!git log -1 --format="commit %H (%cd)"

In [ ]:
!pip install -q nvidia-ml-py pandas matplotlib pytest

import triton
print("torch", torch.__version__, "| triton", triton.__version__)

## 3. Correctness tests

Compares both kernels with PyTorch across awkward shapes, fp16 and fp32,
and every launch config the benchmark sweeps. Don't trust benchmark numbers
from a run where this fails.

In [ ]:
!python -m pytest -q

## 4. Timing benchmark

Sweeps row width N, `BLOCK_SIZE` and `num_warps` for the Triton kernels, and
compares with PyTorch eager and `torch.compile`. Each configuration is checked
for correctness before it is timed. Results go to timestamped CSVs in `results/`.

Use `--quick` for a short smoke run first if you like. The full run is what
the README reports.

In [ ]:
# !python -m benchmarks.bench --quick
!python -m benchmarks.bench

## 5. Energy benchmark (optional)

Uses NVML's cumulative energy counter, with idle power subtracted. If the GPU
or VM does not expose that counter, the script prints why and skips; that
is a normal outcome on some cloud GPUs, not an error.

On shared Colab hardware, idle power and clocks can drift during a run. The
script logs idle power at both ends, plus SM clock and temperature per
measurement, so this can be checked in the CSV.

In [ ]:
!python -m benchmarks.energy

## 6. Plots

In [ ]:
!python -m benchmarks.plot

from IPython.display import Image, display
import glob
for path in sorted(glob.glob("plots/*.png")):
    print(path)
    display(Image(path))

## 7. Save the results

Records the exact environment next to the CSVs, then zips `results/` and
`plots/` for download. To add them to the repo, unzip into your local clone
and commit them; the README should only cite numbers from those files.

In [ ]:
import datetime, subprocess
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
with open(f"results/env_{stamp}.txt", "w") as f:
    for cmd in ["git log -1 --format=%H", "nvidia-smi -q", "pip freeze"]:
        f.write(f"$ {cmd}\n")
        f.write(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout + "\n")

archive = f"/content/triton_results_{stamp}.zip"
!zip -q -r {archive} results plots
print("saved", archive)

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass  # not on Colab; the zip is on disk at the path above